# 01 — Data exploration

Exploration and data quality checks for NYC Yellow Taxi trips (Jan–Mar 2024).
This notebook is a one-off investigation; recurring checks become part of the pipeline in stage 3.

In [8]:
# Imports

import sys
import pandas as pd
from pathlib import Path

In [9]:
# Project root

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Making src/ importable and starting Spark

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from spark_session import get_spark
spark = get_spark()

## 2.1 Schema comparison across months

TLC warns that schemas may change between periods. Each month is read separately before any union.

In [27]:
month_files = sorted(RAW_DIR.glob("*.parquet"))

column_list = []
parquet_name = []
column_size_by_file = []
for path in month_files:
    trips_month = spark.read.parquet(str(path))
    column_list.append(trips_month.dtypes)
    column_size_by_file.append(len(trips_month.columns))
    parquet_name.append(path.stem)

if len(set(column_size_by_file)) != 1:
    raise ValueError("Files have different number of columns.")
else:
    print("All files have the same number of columns.")
    column_size = column_size_by_file[0]


All files have the same number of columns.


In [ ]:
df_file_columns = pd.DataFrame(data=column_list, columns=range(0,column_size),
                         index=parquet_name)

# Checking existence of columns with different names or data_types
has_nonunique_col = False
nonunique_cols = []
for column in range(0, column_size):
    if (not(df_file_columns[column].nunique())):
        has_nonunique_col = True
        nonunique_cols.append(column)

if has_nonunique_col:
    print(f"Non unique cols: {nonunique_cols}")
else:
    print(f"All files have the same column names and data types.")



All files have the same column names and data types.
